# Job Application Agent using Multiple Agents

 * [Runtime Context](#runtime_context)
 * [MCP Server integration with Agents¶](#mcp_server)
 * [Dynamic Tool Calling](#dynamic_tool_calling)
 * [Create multiple Agents As Tools](#create_multiple_agents)

In [1]:
from dotenv import load_dotenv

load_dotenv()

MODEL = "gpt-5-nano"

### MCP Server integration with Agents
<a id='mcp_server'></a>

In [2]:
from langchain_mcp_adapters.client import MultiServerMCPClient

# Run uvx mcp-server-linkedin@latest --login to use the below MCP server
client = MultiServerMCPClient(
    {
        "mcp-server-linkedin": {
            "transport": "stdio",
            "command": "uvx",
            "args": ["mcp-server-linkedin@latest"],
            "env": {"UV_HTTP_TIMEOUT": "300"}
        }
    }
)

mcp_tools = await client.get_tools()
job_mcp_tools = [t for t in mcp_tools if 'job' in t.name]

### Dynamic Tool calling

<a id='dynamic_tool_calling'></a>

In [3]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable

@wrap_model_call
async def dynamic_tool_call(request: ModelRequest, handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:

    """Dynamically call tools based on the runtime context"""

    platform = request.runtime.context.platform
    print("Platform: " + platform)
    
    if platform and platform.lower() != "linkedin":
        tools = [job_finder, display_jobs, apply_jobs] 
    else:
        tools = job_mcp_tools + [apply_jobs, display_jobs, save_mcp_job_results_to_csv]
    request = request.override(tools=tools) 

    return await handler(request)

### Runtime Context

<a id='runtime_context'></a>

In [4]:
from dataclasses import dataclass

@dataclass
class JobSearchContext:
    platform: str
    role: str
    city: str

In [5]:
import os
import requests
from bs4 import BeautifulSoup
from langchain.tools import tool
from job_scraper import scrape_jobs_to_csv
from linkedin_scraper import scrape_linkedin_jobs_to_csv
from langgraph.types import Command
from langchain.messages import ToolMessage
from langchain.tools import tool, ToolRuntime

@tool
async def job_finder(runtime: ToolRuntime, platform: str, role: str, city: str, radius: int = 30) -> str:
    """Find jobs based on platform the user wants to search jobs on and create a csv file with list of jobs to apply to"""
    
    csv_filename = "job_details.csv"
    csv_path = os.path.join(os.getcwd(), csv_filename)
    
    if platform.lower() == "stepstone":
        
        scrape_jobs_to_csv(
            f"https://www.stepstone.de/work/{role.strip()}/in-{city.lower()}?radius={radius}",
            csv_filename,
            max_jobs=3 # 25 jobs per page
        )
    else:
        await scrape_linkedin_jobs_to_csv(role, city, csv_path)
        
    return Command(update={
        "pathToJobsCsv": csv_path,
        "messages": [ToolMessage(
            f"Successfully updated csv file at path: {csv_path}. Next step is to display all the jobs",
            tool_call_id=runtime.tool_call_id,
        )]})

In [6]:
@tool
def display_jobs(runtime: ToolRuntime) -> str:
    """ Display the title, company, and location of every job found so far """
    try:
        csv_path = runtime.state["pathToJobsCsv"]
    except KeyError:
        return "No CSV file path key found"    
        
    with open(csv_path, newline="", encoding="utf-8") as f:
        rows = list(csv.DictReader(f))

    columns = ["title", "company", "location"]
    widths = {col: max(len(col), *(len(row[col]) for row in rows)) for col in columns}

    def format_row(values):
        return " | ".join(str(v).ljust(widths[col]) for col, v in zip(columns, values))

    lines = [format_row(columns), "-+-".join("-" * widths[col] for col in columns)]
    lines.extend(format_row([row[col] for col in columns]) for row in rows)

    return "\n".join(lines)

In [7]:
@tool
def apply_jobs(runtime: ToolRuntime) -> str:
    """ Apply to jobs listed in csv post confirmation on all the jobs from human"""
    return ToolMessage("Applied to all the jobs", tool_call_id=runtime.tool_call_id)

In [8]:
from linkedin_scraper import linkedin_tool_messages_to_csv

@tool
def save_mcp_job_results_to_csv(runtime: ToolRuntime) -> str:
    """Call this after using LinkedIn's search_jobs/get_job_details tools directly, to consolidate \
       their raw results into a jobs csv with the same format as the StepStone scraper."""
    
    csv_filename = "job_details.csv"
    csv_path = os.path.join(os.getcwd(), csv_filename)
    row_count = linkedin_tool_messages_to_csv(runtime.state["messages"], csv_path)

    if row_count == 0:
        return "No search_jobs/get_job_details results found yet — call those tools first."

    return Command(update={
        "pathToJobsCsv": csv_path,
        "messages": [ToolMessage(
            f"Consolidated {row_count} LinkedIn job(s) into {csv_path}.",
            tool_call_id=runtime.tool_call_id,
        )],
    })

In [9]:
import asyncio
import csv
from langchain.messages import HumanMessage
from langchain.chat_models import init_chat_model

@tool
async def analyze_resume_and_make_suggestions(runtime: ToolRuntime) -> str:
    """Fetch job description and add changes to resume(passed as pdf) by tailoring it to each job description and save it to csv for human approval"""
    try:
        csv_path = runtime.state["pathToJobsCsv"]
        pdf_base64 = runtime.state["pdfBase64"]
    except KeyError:
        return "No CSV file path key found"   
        
    with open(csv_path, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        rows = list(reader)
        fieldnames = reader.fieldnames + ["resume_corrections"] + ["apply"]

    llm = init_chat_model(MODEL)

    async def get_suggestions(row):
        message = HumanMessage(
            content=[
                {
                    "type": "text",
                    "text": (
                        "You are a resume coach. Compare the attached resume to the job descriptions "
                        "and suggest specific, concise corrections or additions to better match the job. "
                        "Match their language, mirror their priorities, and keep it to one page. "
                        "Don’t make anything up — only use what’s already in my resume, just position it better."
                        "Return only a short bulleted list of suggestions.\n\n"
                        f"Job Description:\n{row['description']}\n"
                    ),
                },
                {
                    "type": "file",
                    "mime_type": "application/pdf",
                    "base64": pdf_base64,
                },
            ]
        )
        response = await llm.ainvoke([message])
        row["resume_corrections"] = response.content.replace("\n", " | ")
        row["apply"] = True
            
    await asyncio.gather(*(get_suggestions(row) for row in rows))

    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)
    
    return ToolMessage(f"""Tailored resume suggestions for all jobs and saved to csv {csv_path}. 
                    Next step: Make corrections to resume and download""", tool_call_id=runtime.tool_call_id)

In [10]:
import asyncio
import csv
from langchain.messages import HumanMessage
from langchain.chat_models import init_chat_model

@tool
async def resume_corrections_and_download(runtime: ToolRuntime) -> str:
    """Apply the suggested resume corrections and save each tailored resume as a new file \
        (never overwriting the original), one per job marked apply=True in the jobs csv."""
    try:
        csv_path = runtime.state["pathToJobsCsv"]
        pdf_base64 = runtime.state["pdfBase64"]
    except KeyError:
        return "No CSV file path key found"   
        
    with open(csv_path, newline="", encoding="utf-8") as f:
        rows = list(csv.DictReader(f))    

    llm = init_chat_model(MODEL)
    saved_files = []
    
    async def apply_suggestions(row):
        if row.get("apply", "").strip().lower() != "true":
            return
            
        message = HumanMessage(
            content=[
                {
                    "type": "text",
                    "text": (
                        "Rewrite the attached resume, applying the suggested corrections below. "
                        "Do not invent new experience or skills — only reorganize, rephrase, and "
                        "emphasize what is already in the original resume, per the suggestions. "
                        "Output the complete rewritten resume as plain text, ready to save as a file. "
                        "Do not include any commentary before or after the resume text.\n\n"
                        f"Suggested changes:\n{row['resume_corrections']}\n"
                    ),
                },
                {
                    "type": "file",
                    "mime_type": "application/pdf",
                    "base64": pdf_base64,
                },
            ]
        )
        response = await llm.ainvoke([message])
        
        filename = f"{row['company']}_{row['title']}_tailored_resume.txt".replace(" ", "_").replace("/", "-")
        filepath = os.path.join(os.getcwd(), filename)
        with open(filepath, "w", encoding="utf-8") as f:
            f.write(response.content)
        saved_files.append(filepath)

    await asyncio.gather(*(apply_suggestions(row) for row in rows))

    return Command(update={
        "tailoredResumeFiles": saved_files,
        "messages": [ToolMessage(
            f"""Saved {len(saved_files)} tailored resume file(s): {', '.join(saved_files)}. 
            Next Step: Apply to jobs which are marked as True in csv column apply""",
            tool_call_id=runtime.tool_call_id,
        )],
    })

### Create Multiple Agents
<a id='create_multiple_agents'></a>


In [11]:
MAIN_AGENT_SYSTEM_PROMPT = """You are a job application assistant. You do not search for jobs or write \
resume content yourself — you delegate to two specialist subagents and relay their results to the user.

## Subagents

- `call_job_search_agent`: delegates to a subagent that searches for jobs on a given platform (LinkedIn \
or StepStone) and can apply to jobs once approved. Use this whenever the user wants to find or apply to jobs.
- `call_resume_handler_agent`: delegates to a subagent that analyzes the user's resume against found job \
descriptions and produces tailored versions. Use this after jobs have been found, whenever the user wants \
resume suggestions or a tailored copy.

## Rules

1. Call `call_job_search_agent` first when the user wants jobs found. Do not call `call_resume_handler_agent` \
before jobs exist — it needs the jobs csv the search subagent produces.
2. Never fabricate job details or resume content yourself. Only relay what the subagents actually returned.
3. Run the workflow end to end without pausing to ask permission between steps — the subagents themselves \
handle any approval that's required.
4. Summarize each subagent's concrete results (job titles/companies, or resume changes made) rather than a \
generic confirmation.
"""

JOB_SEARCH_AGENT_SYSTEM_PROMPT = """You search for jobs on the platform, role, and location you're given, \
and can apply to jobs once approved.

## Tools

- `job_finder`: search for jobs on the given platform (defaults to LinkedIn if none specified) and role/location.
- `display_jobs`: show the jobs found so far.
- `apply_jobs`: apply to the jobs marked for application in the jobs csv.

## Rules

1. Always call `job_finder` first, then `display_jobs`, each as its own step.
2. Call `apply_jobs` directly when asked to apply — the system automatically pauses for human approval the \
moment you call it. Do not ask for confirmation in chat first; that skips the approval mechanism entirely.
3. Never fabricate job listings or details — only report what `job_finder`/`display_jobs` actually returned.
"""

RESUME_HANDLER_AGENT_SYSTEM_PROMPT = """You analyze a resume against job descriptions and produce tailored \
versions, without ever modifying the original resume.

## Tools

- `analyze_resume_and_make_suggestions`: compare the resume to each job description found so far and save \
suggested corrections to the jobs csv.
- `resume_corrections_and_download`: apply the saved suggestions to a *new* file per job marked for \
application — never overwrite the original resume.

## Rules

1. Call `analyze_resume_and_make_suggestions` first, before `resume_corrections_and_download`.
2. Only apply suggestions to jobs explicitly marked `apply=true` in the csv — never assume a job should be \
included.
3. Never invent resume content — only reorganize and rephrase what's already in the original resume, per \
the generated suggestions.
"""

In [12]:
from langchain.agents import AgentState

class JobApplicationAgentState(AgentState):
    pathToJobsCsv: str
    pdfBase64: str
    tailoredResumeFiles: list
    
class JobSearchAgentState(AgentState):
    pathToJobsCsv: str

class ResumeHandlerAgentState(AgentState):
    pathToJobsCsv: str
    pdfBase64: str
    tailoredResumeFiles: list
    

In [13]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver  
from langchain.agents.middleware import HumanInTheLoopMiddleware

# create subagents

job_search_agent = create_agent(
    model=MODEL,
    system_prompt=JOB_SEARCH_AGENT_SYSTEM_PROMPT,
    tools=[job_finder, display_jobs, apply_jobs, save_mcp_job_results_to_csv] + job_mcp_tools,
    state_schema=JobSearchAgentState,
    context_schema=JobSearchContext,
    checkpointer=InMemorySaver(),
    middleware=[dynamic_tool_call]
)

resume_handler_agent = create_agent(
    model=MODEL,
    system_prompt=RESUME_HANDLER_AGENT_SYSTEM_PROMPT,
    tools=[analyze_resume_and_make_suggestions, resume_corrections_and_download],
    state_schema=ResumeHandlerAgentState,
    checkpointer=InMemorySaver(),
    middleware=[
            HumanInTheLoopMiddleware(
                interrupt_on={
                    "resume_corrections_and_download": True
                }
            )
        ],
)


In [14]:
from typing import Literal

@tool
async def call_job_search_agent(platform: Literal["stepstone", "linkedin"], role: str, city: str, runtime: ToolRuntime) -> str:
    """Call the job search subagent to find jobs and optionally apply to them."""
    config = {"configurable": {"thread_id": f"{runtime.config['configurable']['thread_id']}-job-search"}}
    response = await job_search_agent.ainvoke({"messages": [HumanMessage(content=f"Find job for {role} in {city} on {platform}")]},
                                              context=JobSearchContext(platform=platform, role=role, city=city),
                                              config=config)

    return Command(update={
        "pathToJobsCsv": response.get("pathToJobsCsv"),
        "messages": [ToolMessage(response["messages"][-1].content, tool_call_id=runtime.tool_call_id)],
    })

@tool
async def call_resume_handler_agent(runtime: ToolRuntime) -> str:
    """Call the resume handler subagent to analyze the resume against found jobs and produce tailored versions."""
    config = {"configurable": {"thread_id" : f"{runtime.config['configurable']['thread_id']}-resume-handler"}}
    response = await resume_handler_agent.ainvoke({"messages": [HumanMessage(content=f"Analyze my resume against the job descriptions and produce tailored versions.")],
                                                     "pdfBase64": runtime.state.get("pdfBase64"),
                                                    "pathToJobsCsv": runtime.state.get("pathToJobsCsv"),
                                                  }, config=config)
    if response.get("__interrupt__"):
        print(response["__interrupt__"]) 
        response = await resume_handler_agent.ainvoke(
            Command( 
                resume={"decisions": [{"type": "approve"}]}
            ), 
            config=config # Same thread ID to resume the paused conversation
        )
        
    return Command(update={
        "tailoredResumeFiles": response.get("tailoredResumeFiles"),
        "messages": [ToolMessage(response["messages"][-1].content, tool_call_id=runtime.tool_call_id)],
    })

## Creating the main agent
job_application_agent = create_agent(
    model=MODEL,
    state_schema=JobApplicationAgentState,
    tools=[call_job_search_agent, call_resume_handler_agent],
    system_prompt=MAIN_AGENT_SYSTEM_PROMPT,
    checkpointer=InMemorySaver())

In [15]:
# File uploader
import base64
from ipywidgets import FileUpload
from IPython.display import display

uploader = FileUpload(accept='.pdf', multiple=False)
display(uploader)

FileUpload(value=(), accept='.pdf', description='Upload')

In [16]:
# Get the first (and only) uploaded file dict
uploaded_file = uploader.value[0]

# This is a memoryview
content_mv = uploaded_file["content"]

# Convert memoryview -> bytes
pdf_bytes = bytes(content_mv)  # or content_mv.tobytes()

# Now base64 encode
pdf_base64 = base64.b64encode(pdf_bytes).decode("utf-8")

In [ ]:
config = {"configurable": {"thread_id": "1"}}

response = await job_application_agent.ainvoke(
    {"messages": [HumanMessage(content="""Find jobs for AI engineer in Berlin on Stepstone, 
                                        update my resume to the job description and apply to jobs""")
                 ],
     "pdfBase64": pdf_base64,
    },
    config=config
)
print(response['messages'][-1].content)

Platform: stepstone


/Users/nirjharijankar/projects/AI/langchain/job_application_agent/.venv/lib/python3.12/site-packages/pydantic/functional_validators.py:835: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=JobSearchContext(platform...ngineer', city='Berlin'), input_type=JobSearchContext])
  function=lambda v, h: h(v), schema=original_schema
/Users/nirjharijankar/projects/AI/langchain/job_application_agent/.venv/lib/python3.12/site-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=JobSearchContext(platform...ngineer', city='Berlin'), input_type=JobSearchContext])
  return self.__pydantic_serializer__.to_python(


[1/3] scraped: AI Engineer
[2/3] scraped: AI Engineer
[3/3] scraped: AI Engineer / Data Scientist (mwd) – Machine Learning & Generative AI
Platform: stepstone


/Users/nirjharijankar/projects/AI/langchain/job_application_agent/.venv/lib/python3.12/site-packages/pydantic/functional_validators.py:835: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=JobSearchContext(platform...ngineer', city='Berlin'), input_type=JobSearchContext])
  function=lambda v, h: h(v), schema=original_schema
/Users/nirjharijankar/projects/AI/langchain/job_application_agent/.venv/lib/python3.12/site-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=JobSearchContext(platform...ngineer', city='Berlin'), input_type=JobSearchContext])
  return self.__pydantic_serializer__.to_python(


Platform: stepstone


/Users/nirjharijankar/projects/AI/langchain/job_application_agent/.venv/lib/python3.12/site-packages/langchain_core/messages/block_translators/openai.py:109: UserWarning: OpenAI may require a filename for file uploads. Specify a filename in the content block, e.g.: {'type': 'file', 'mime_type': '...', 'base64': '...', 'filename': 'my-file.pdf'}. Using placeholder filename 'LC_AUTOGENERATED'.
  warnings.warn(


[Interrupt(value={'action_requests': [{'name': 'resume_corrections_and_download', 'args': {}, 'description': 'Tool execution requires approval\n\nTool: resume_corrections_and_download\nArgs: {}'}], 'review_configs': [{'action_name': 'resume_corrections_and_download', 'allowed_decisions': ['approve', 'edit', 'reject', 'respond']}]}, id='7717c06605a39c429115b8a9be1c1efc')]


/Users/nirjharijankar/projects/AI/langchain/job_application_agent/.venv/lib/python3.12/site-packages/langchain_core/messages/block_translators/openai.py:109: UserWarning: OpenAI may require a filename for file uploads. Specify a filename in the content block, e.g.: {'type': 'file', 'mime_type': '...', 'base64': '...', 'filename': 'my-file.pdf'}. Using placeholder filename 'LC_AUTOGENERATED'.
  warnings.warn(
